## Terrain-following firn model

This notebook implements a vertically resolved firn model on the fixed computational domain

$$
D=\Omega\times[0,1],
$$

where $\Omega$ is the horizontal domain and

$$
\zeta=\frac{z-z_b}{h}
$$

is the terrain-following vertical coordinate.

The physical firn thickness is

$$
h(\mathbf{x},t)=z_s(\mathbf{x},t)-z_b(\mathbf{x},t),
$$

where $z_s$ and $z_b$ are the physical surface and basal elevations.

---

### Notation

Horizontal position:

$$
\mathbf{x}=(x,y).
$$

The computational gradient is

$$
\nabla_\xi
=
\left(\nabla_h,\partial_\zeta\right),
$$

where $\nabla_h$ denotes horizontal differentiation at fixed $\zeta$.

The model variables are:

- $\rho(\mathbf{x},\zeta,t)$: firn density,
- $h(\mathbf{x},t)$: physical firn thickness,
- $\mathbf{u}(\mathbf{x},\zeta,t)$: prescribed horizontal velocity,
- $\omega(\mathbf{x},\zeta,t)$: grid-relative terrain-following vertical velocity,
- $\mathbf{U}=(\mathbf{u},\omega)$: full velocity in computational coordinates.

The terrain-following vertical velocity is defined by

$$
\omega=\frac{D\zeta}{Dt}.
$$

If $w$ denotes the physical vertical velocity, then

$$
\omega
=
\frac{
w
-\partial_t z_b
-\mathbf{u}\cdot\nabla_h z_b
-\zeta\left(
\partial_t h+\mathbf{u}\cdot\nabla_h h
\right)
}{h}.
$$

Thus $\omega$ has units of inverse time and measures the fraction of the firn-column thickness traversed per unit time relative to the moving terrain-following grid.

The firn compaction law is

$$
C=C(\rho,\ldots),
$$

with units of density per unit time. Positive $C$ corresponds to increasing density:

$$
\frac{D\rho}{Dt}=C.
$$

Surface and basal mass-balance rates are denoted

$$
\dot a_s,
\qquad
\dot a_b,
$$

with units of length per unit time. The sign convention is:

- $\dot a_s>0$: accumulation adds firn at the surface,
- $\dot a_b>0$: firn is converted to glacier ice at the base.

The prescribed density of newly accumulated material is $\rho_s$.

Define the conserved density variable

$$
q:=h\rho.
$$

---

### Continuous governing equations

#### 1. Firn mass conservation

The conservative density equation on the fixed computational domain is

$$
\partial_t(h\rho)
+
\nabla_\xi\cdot(h\rho\mathbf{U})
=
0.
$$

In terms of $q=h\rho$,

$$
\boxed{
\partial_t q+\nabla_\xi\cdot(q\mathbf{U})=0.
}
$$

Although the equation is written in terms of $q$, the numerical unknown is $\rho$, with

$$
q=h\rho
$$

defined symbolically.

---

#### 2. Compaction and transformed velocity divergence

Expanding the conservative mass equation gives

$$
0
=
h\frac{D\rho}{Dt}
+
\rho
\left[
\partial_t h
+
\nabla_\xi\cdot(h\mathbf{U})
\right].
$$

Using $D\rho/Dt=C$ therefore gives

$$
\boxed{
\frac{1}{h}
\left[
\partial_t h
+
\nabla_\xi\cdot(h\mathbf{U})
\right]
=
-\frac{C(\rho)}{\rho}.
}
$$

Since $h$ is independent of $\zeta$ and

$$
\mathbf{U}=(\mathbf{u},\omega),
$$

this becomes

$$
\boxed{
\partial_t h
+
\nabla_h\cdot(h\mathbf{u})
+
h\partial_\zeta\omega
+
h\frac{C(\rho)}{\rho}
=
0.
}
$$

Equivalently,

$$
\partial_\zeta\omega
=
-\frac{1}{h}\partial_t h
-\frac{1}{h}\nabla_h\cdot(h\mathbf{u})
-\frac{C(\rho)}{\rho}.
$$

The equation determines only the vertical derivative of $\omega$. One vertical boundary value is therefore imposed to remove the $\zeta$-independent nullspace.

---

#### 3. Thickness evolution

Define the vertically averaged horizontal velocity

$$
\overline{\mathbf{u}}
=
\int_0^1\mathbf{u}\,d\zeta.
$$

Because the computational vertical interval has unit length, this integral is also the vertical average.

Define the column-integrated fractional compaction rate

$$
\mathcal{D}
=
\int_0^1\frac{C(\rho)}{\rho}\,d\zeta.
$$

The conservative thickness equation is

$$
\boxed{
\partial_t h
+
\nabla_h\cdot(h\overline{\mathbf{u}})
=
\dot a_s-\dot a_b-h\mathcal{D}.
}
$$

Thus thickness changes through:

- horizontal transport,
- surface accumulation,
- basal conversion,
- compaction.

Integrating the transformed compaction equation vertically gives

$$
\partial_t h
+
\nabla_h\cdot(h\overline{\mathbf{u}})
+
h(\omega_s-\omega_b)
+
h\mathcal{D}
=
0.
$$

Combining this with the thickness equation gives the boundary compatibility condition

$$
h(\omega_s-\omega_b)
=
-\dot a_s+\dot a_b.
$$

---

### Boundary and initial conditions

#### Density boundary fluxes

At the firn surface, positive accumulation is an inward mass flux. The outward numerical mass flux is therefore

$$
\boxed{
\mathcal{F}_s=-\rho_s\dot a_s.
}
$$

At the firn base, positive basal conversion is an outward mass flux:

$$
\boxed{
\mathcal{F}_b=+\rho_b\dot a_b,
}
$$

where

$$
\rho_b=\rho|_{\zeta=0}.
$$

On lateral inflow boundaries, the incoming density $\rho_{\mathrm{in}}$ must be prescribed.

The surface formula above assumes accumulation. If negative surface mass balance is allowed, the surface boundary flux must instead use an appropriate inflow/outflow rule.

---

#### Terrain-following vertical-velocity conditions

Because $\omega=D\zeta/Dt$ is measured relative to the moving terrain-following grid, motion of $z_s$ and $z_b$ is already included in its definition.

At the surface,

$$
h\omega_s
=
w_s
-\partial_t z_s
-\mathbf{u}_s\cdot\nabla_h z_s.
$$

The physical surface kinematic condition is

$$
\partial_t z_s
+
\mathbf{u}_s\cdot\nabla_h z_s
-
w_s
=
\dot a_s.
$$

Therefore,

$$
\boxed{
h\omega_s=-\dot a_s,
}
$$

or

$$
\boxed{
\omega_s=-\frac{\dot a_s}{h}.
}
$$

Likewise, at the base,

$$
h\omega_b
=
w_b
-\partial_t z_b
-\mathbf{u}_b\cdot\nabla_h z_b.
$$

Using the basal kinematic condition

$$
\partial_t z_b
+
\mathbf{u}_b\cdot\nabla_h z_b
-
w_b
=
\dot a_b
$$

gives

$$
\boxed{
h\omega_b=-\dot a_b,
}
$$

or

$$
\boxed{
\omega_b=-\frac{\dot a_b}{h}.
}
$$

These relations hold whether or not the physical surface or base is stationary.

Because the equation for $\omega$ is first order in $\zeta$, only one of these values should be imposed directly. For example, one may impose

$$
\omega|_{\zeta=0}
=
-\frac{\dot a_b}{h}.
$$

The surface value

$$
\omega|_{\zeta=1}
=
-\frac{\dot a_s}{h}
$$

then follows from compatibility between the compaction equation and the thickness equation and can be used as a numerical consistency check.

---

#### Initial conditions

The prognostic fields require

$$
\rho(\mathbf{x},\zeta,0)
=
\rho_0(\mathbf{x},\zeta)
$$

and

$$
h(\mathbf{x},0)
=
h_0(\mathbf{x}).
$$

---

### Finite-element spaces

The intended spatial discretization is

$$
\rho_h\in DG_k(D),
$$

$$
\omega_h\in CG_{k+1}
\quad\text{in the vertical direction},
$$

and

$$
h_h\in DG_0(\Omega).
$$

The test spaces are

$$
\phi_h\in DG_k(D)
$$

for density and

$$
\psi_h\in DG_k(D)
$$

for the transformed velocity-divergence equation.

The pairing

$$
\omega_h\in CG_{k+1},
\qquad
\psi_h\in DG_k
$$

is used because

$$
\partial_\zeta:
CG_{k+1}\longrightarrow DG_k.
$$

The thickness field has one value per horizontal cell and is evolved using a finite-volume balance implemented through Firedrake's $DG_0$ space.

---

### Discrete density equation

Let

$$
q=h\rho,
\qquad
q_{\mathrm{old}}
=
h_{\mathrm{old}}\rho_{\mathrm{old}}.
$$

For backward Euler,

$$
\partial_t q
\approx
\frac{q-q_{\mathrm{old}}}{\Delta t}.
$$

On an interior facet, let $+$ and $-$ denote the two adjacent cell traces. Let $\mathbf{n}^+$ point outward from the $+$ cell and define

$$
U_n
=
\frac{\mathbf{U}^++\mathbf{U}^-}{2}
\cdot\mathbf{n}^+.
$$

The upwind mass flux oriented outward from the $+$ cell is

$$
\boxed{
\mathcal{F}_{\mathrm{int}}
=
q^+\max(U_n,0)
+
q^-\min(U_n,0).
}
$$

If $\mathbf{U}$ is continuous, the averaging is redundant.

The backward-Euler DG residual is

$$
\begin{aligned}
F_\rho
={}&
\int_D
\frac{q-q_{\mathrm{old}}}{\Delta t}\phi
-
\int_D
q\mathbf{U}\cdot\nabla_\xi\phi
\\
&+
\int_{\Gamma_{\mathrm{int}}}
(\phi^+-\phi^-)\mathcal{F}_{\mathrm{int}}
\\
&+
\int_{\Gamma_{\mathrm{lat}}}
\phi
\left[
q\max(\mathbf{U}\cdot\mathbf{n},0)
+
q_{\mathrm{in}}
\min(\mathbf{U}\cdot\mathbf{n},0)
\right]
\\
&-
\int_{\Gamma_s}
\rho_s\dot a_s\phi
+
\int_{\Gamma_b}
\rho_b\dot a_b\phi
=
0,
\end{aligned}
$$

where

$$
q_{\mathrm{in}}=h\rho_{\mathrm{in}}.
$$

On an extruded Firedrake mesh, interior facets include both:

- vertical facets between neighboring horizontal columns,
- horizontal facets between neighboring vertical layers.

---

### Discrete transformed velocity-divergence equation

The velocity-divergence residual is not integrated by parts.

For backward Euler, define

$$
\dot h
\approx
\frac{h-h_{\mathrm{old}}}{\Delta t}.
$$

For every

$$
\psi\in DG_k(D),
$$

solve

$$
\boxed{
F_\omega
=
\int_D
\left[
\dot h
+
h\partial_\zeta\omega
+
\nabla_h\cdot(h\mathbf{u})
+
h\frac{C(\rho)}{\rho}
\right]\psi
=
0.
}
$$

One vertical boundary value for $\omega$ is imposed to remove the nullspace. For example,

$$
\omega|_{\zeta=0}
=
-\frac{\dot a_b}{h}.
$$

The computed surface value should satisfy

$$
\omega|_{\zeta=1}
=
-\frac{\dot a_s}{h}
$$

up to discretization and nonlinear-solver error.

---

### DG0 thickness update

For each horizontal cell $K$, the thickness is constant:

$$
h(\mathbf{x},t)=h_K(t),
\qquad
\mathbf{x}\in K.
$$

The governing finite-volume balance is

$$
\boxed{
|K|\frac{dh_K}{dt}
+
\sum_{e\subset\partial K}
\int_e\mathcal{F}_{K,e}\,ds
=
\int_K
\left(
\dot a_s-\dot a_b-h_K\mathcal{D}
\right)
d\mathbf{x}.
}
$$

For an interior horizontal facet shared by cells $+$ and $-$, define

$$
\overline{u}_n
=
\frac{
\overline{\mathbf{u}}^+
+
\overline{\mathbf{u}}^-
}{2}
\cdot\mathbf{n}^+.
$$

The upwind thickness flux is

$$
\boxed{
\mathcal{F}_{h,\mathrm{int}}
=
h^+\max(\overline{u}_n,0)
+
h^-\min(\overline{u}_n,0).
}
$$

On a horizontal exterior boundary,

$$
\boxed{
\mathcal{F}_{h,\mathrm{ext}}
=
h\max(\overline{\mathbf{u}}\cdot\mathbf{n},0)
+
h_{\mathrm{in}}
\min(\overline{\mathbf{u}}\cdot\mathbf{n},0).
}
$$

The backward-Euler cell update is

$$
\boxed{
|K|
\frac{h_K-h_{K,\mathrm{old}}}{\Delta t}
+
\sum_{e\subset\partial K}
\int_e\mathcal{F}_{K,e}\,ds
-
\int_K\dot a_s\,d\mathbf{x}
+
\int_K\dot a_b\,d\mathbf{x}
+
\int_Kh_K\mathcal{D}\,d\mathbf{x}
=
0.
}
$$

In Firedrake, this finite-volume equation is assembled using a horizontal $DG_0$ function space.

---

### Model inputs still to be specified

The governing system additionally requires:

- a particular compaction law $C(\rho,\ldots)$,
- a basal conversion law for $\dot a_b$,
- prescribed horizontal velocity $\mathbf{u}$,
- surface accumulation $\dot a_s$,
- surface density $\rho_s$,
- lateral inflow density $\rho_{\mathrm{in}}$,
- lateral inflow thickness $h_{\mathrm{in}}$ if the horizontal domain is open,
- initial fields $\rho_0$ and $h_0$,
- one imposed vertical boundary value for $\omega$.

In [1]:
from firedrake import *
from irksome import BackwardEuler, Dt, MeshConstant, TimeStepper

In [2]:
# Computational domain: Omega x [0, 1]
n_horizontal = 1
n_vertical = 80

base_mesh = UnitIntervalMesh(n_horizontal)

mesh = ExtrudedMesh(
    base_mesh,
    layers=n_vertical,
    layer_height=1.0 / n_vertical,
)

zeta_axis = mesh.geometric_dimension - 1
zeta = SpatialCoordinate(mesh)[zeta_axis]

In [3]:
degree = 1

V_rho = FunctionSpace(
    mesh,
    "DG", degree,
    vfamily="DG", vdegree=degree,
)

V_h = FunctionSpace(
    mesh,
    "DG", 0,
    vfamily="R", vdegree=0,
)

V_p = FunctionSpace(
    mesh,
    "DG", degree,
    vfamily="CG", vdegree=degree + 1,
)

# a_b has the same horizontal-only structure as h
V_ab = V_h

W = V_rho * V_h * V_p * V_ab

In [4]:
state = Function(W, name="firn_state")

rho, h, p, a_b = split(state)
phi, eta, psi, mu = TestFunctions(W)

rho_f, h_f, p_f, ab_f = state.subfunctions

In [5]:
omega = p / h

In [6]:
rho_i_value = 917.0       # kg m^-3
rho_s_value = 300.0       # kg m^-3
a_s_value = 0.30          # m yr^-1
h_ref_value = 60.0        # m
c_value = 0.01            # m^-1

rho_i = Constant(rho_i_value)
rho_s = Constant(rho_s_value)
a_s = Constant(a_s_value)
h_ref = Constant(h_ref_value)
c = Constant(c_value)

In [8]:
rho_in = Constant(rho_s_value)
h_in = Constant(h_ref_value)

q_in = h_in * rho_in

In [9]:
n_horiz = mesh.geometric_dimension - 1
zeta_axis = mesh.geometric_dimension - 1

V_u = FunctionSpace(
    mesh,
    "CG", 1,
    vfamily="R", vdegree=0,
)

u_components = [
    Function(V_u, name=f"u_{i}") for i in range(n_horiz)
]

u_bar_components = [
    Function(V_u, name=f"u_bar_{i}") for i in range(n_horiz)
]

for component in u_components:
    component.assign(0.0)

for component in u_bar_components:
    component.assign(0.0)

u = as_vector(u_components)
u_bar = as_vector(u_bar_components)

In [10]:
def grad_h(f):
    """Gradient with respect to horizontal coordinates at fixed zeta."""
    return as_vector([f.dx(i) for i in range(n_horiz)])


def div_h(v):
    """Horizontal divergence at fixed zeta."""
    return sum(v[i].dx(i) for i in range(n_horiz))

In [11]:
normal = FacetNormal(mesh)
normal_h = as_vector([normal[i] for i in range(n_horiz)])

In [12]:
C = c * a_s * max_value(rho_i - rho, 0.0)
fractional_compaction = C / rho

In [13]:
q = h * rho

In [14]:
u_n = dot(avg(u), normal_h("+"))

q_flux_horizontal = (
    q("+") * max_value(u_n, 0.0)
    + q("-") * min_value(u_n, 0.0)
)

In [15]:
p_n = avg(p) * normal("+")[zeta_axis]

q_flux_vertical = (
    rho("+") * max_value(p_n, 0.0)
    + rho("-") * min_value(p_n, 0.0)
)

In [16]:
u_n_external = dot(u, normal_h)

q_flux_external = (
    q * max_value(u_n_external, 0.0)
    + q_in * min_value(u_n_external, 0.0)
)

In [17]:
F_rho = (
    # Time derivative
    Dt(q) * phi * dx

    # Cell-interior transport
    - q * dot(u, grad_h(phi)) * dx
    - rho * p * phi.dx(zeta_axis) * dx

    # vertical facets between columns
    + (phi("+") - phi("-"))
      * q_flux_horizontal * dS_v

    # horizontal facets between layers
    + (phi("+") - phi("-"))
      * q_flux_vertical * dS_h

    # Lateral exterior boundary
    + phi * q_flux_external * ds_v

    # Surface accumulation: outward flux = -rho_s a_s
    - rho_s * a_s * phi * ds_t

    # Basal conversion: outward flux = +rho_b a_b
    + rho * a_b * phi * ds_b
)

In [18]:
u_bar_n = dot(avg(u_bar), normal_h("+"))

h_flux_horizontal = (
    h("+") * max_value(u_bar_n, 0.0)
    + h("-") * min_value(u_bar_n, 0.0)
)

In [19]:
u_bar_n_external = dot(u_bar, normal_h)

h_flux_external = (
    h * max_value(u_bar_n_external, 0.0)
    + h_in * min_value(u_bar_n_external, 0.0)
)

In [20]:
F_h = (
    Dt(h) * eta * dx

    - h * dot(u_bar, grad_h(eta)) * dx

    + (eta("+") - eta("-"))
      * h_flux_horizontal * dS_v

    + eta * h_flux_external * ds_v

    + (-a_s + a_b) * eta * dx

    + h * fractional_compaction * eta * dx
)

In [21]:
F_p = (
    (
        Dt(h)
        + div_h(h * u)
        + p.dx(zeta_axis)
        + h * fractional_compaction
    )
    * psi
    * dx
)

In [22]:
basal_conversion_law = (
    a_c
    * max_value(rho - rho_c, 0.0)
    / (rho_i - rho_c)
)

F_basal_law = (
    (a_b - basal_conversion_law)
    * mu
    * ds_b
)

In [23]:
F_fixed_height = (
    (h - h_ref)
    * mu
    * dx
)

In [24]:
fixed_height = True

if fixed_height:
    F_closure = F_fixed_height
else:
    F_closure = F_basal_law

In [25]:
F = F_rho + F_h + F_p + F_closure

In [26]:
bc_p_surface = DirichletBC(
    W.sub(2),
    -a_s,
    "top",
)

bcs = [bc_p_surface]

In [27]:
rho_initial = (
    rho_s_value
    + 100.0 * (1.0 - zeta)
)

rho_f.interpolate(rho_initial)
h_f.assign(h_ref_value)

# Initial guesses for the algebraic variables.
p_f.assign(-a_s_value)
ab_f.assign(a_s_value)

Coefficient(WithGeometry(IndexedProxyFunctionSpace(<firedrake.mesh.ExtrudedMeshTopology object at 0x7895a7058a40>, TensorProductElement(FiniteElement('Discontinuous Lagrange', interval, 0), FiniteElement('Real', interval, 0), cell=TensorProductCell(interval, interval)), name=None, index=3, component=None), Mesh(VectorElement(TensorProductElement(FiniteElement('Lagrange', interval, 1), FiniteElement('Lagrange', interval, 1), cell=TensorProductCell(interval, interval)), dim=2), 3)), 17)

In [28]:
MC = MeshConstant(mesh)

time = MC.Constant(0.0)
dt = MC.Constant(0.05)      # years

method = BackwardEuler()

In [34]:
solver_parameters = {
    "mat_type": "aij",

    "snes_type": "newtonls",
    "snes_monitor": None,
    "snes_converged_reason": None,
    "snes_rtol": 1.0e-9,
    "snes_atol": 1.0e-10,

    "ksp_type": "preonly",
    "ksp_converged_reason": None,

    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "mat_mumps_icntl_24": 1,   # report null pivots
}

In [35]:
stepper = TimeStepper(
    F,
    method,
    time,
    dt,
    state,
    bcs=bcs,
    stage_type="value",
    solver_parameters=solver_parameters,
)

In [36]:
final_time = 20.0
step = 0

domain_volume = assemble(Constant(1.0) * dx(domain=mesh))

while float(time) < final_time:
    print(step)
    if float(time) + float(dt) > final_time:
        dt.assign(final_time - float(time))

    stepper.advance()
    time.assign(float(time) + float(dt))
    step += 1

    if step % 20 == 0:
        mean_h = assemble(h * dx) / domain_volume
        mean_ab = assemble(a_b * dx) / domain_volume
        total_mass = assemble(h * rho * dx)

        print(
            f"t = {float(time):7.3f} yr, "
            f"h = {mean_h:9.5f} m, "
            f"a_b = {mean_ab:9.5f} m/yr, "
            f"M = {total_mass:12.5e}"
        )

0
  0 SNES Function norm 8.513864222666e-02
    Linear firedrake_1_ solve converged due to CONVERGED_ITS iterations 1
  1 SNES Function norm 4.406150450457e-01
    Linear firedrake_1_ solve converged due to CONVERGED_ITS iterations 1
  2 SNES Function norm 1.124010596083e-07
    Linear firedrake_1_ solve converged due to CONVERGED_ITS iterations 1
  3 SNES Function norm 9.490599471365e-14
  Nonlinear firedrake_1_ solve converged due to CONVERGED_FNORM_ABS iterations 3
1
  0 SNES Function norm 6.059735362229e+00
    Linear firedrake_1_ solve converged due to CONVERGED_ITS iterations 1
  1 SNES Function norm 7.761198718473e-05
    Linear firedrake_1_ solve converged due to CONVERGED_ITS iterations 1
  2 SNES Function norm 1.638399157718e-10
  Nonlinear firedrake_1_ solve converged due to CONVERGED_FNORM_RELATIVE iterations 2
2
  0 SNES Function norm 6.227784862804e+00
    Linear firedrake_1_ solve converged due to CONVERGED_ITS iterations 1
  1 SNES Function norm 5.676656633504e-05
    L

ConvergenceError: Nonlinear solve failed to converge after 50 nonlinear iterations.
Reason:
   DIVERGED_MAX_IT

In [33]:
print("p-space dimension:", V_p.dim())
print("surface BC nodes:", len(bc_p_surface.nodes))
print(bc_p_surface.nodes)

p-space dimension: 322
surface BC nodes: 2
[320 321]
